# AutoGen 实战

## Multi-Agent Frameworks — AutoGen 与 CrewAI 实战指南

GraphFlow · SelectorGroupChat · Memory


> ⚠️ **前置**：请先运行 [`03_环境初始化与MAS基础.ipynb`](03_环境初始化与MAS基础.ipynb) 完成依赖安装与环境变量配置。


## 📖 讲义
### 课程资料

2.2 AutoGen 框架要点（概念层面）
定位：AutoGen 强调用 DSL/模板快速生成 Agent（或对话型 agent）并运行协作流程，适合快速原型与教学演示。

核心功能：Agent 定义（name、能力、tools）、对话/计划生成器、消息传递机制、可视化 trace/debug 支持。

使用场景：任务分解、角色扮演式协作（例如“研究员/写作者/审校者”流水线）、交互式调试。

---

### 幻灯片

# Multi-Agent Frameworks
## AutoGen 与 CrewAI 实战指南

课程学习资料 · 面向零基础 AI Engineer 训练营

含 LangGraph 对比与选型 · 120 分钟 · 88 页

---

## 二、AutoGen：定位与设计哲学

- AutoGen 是用于构建 AI agents 和应用的框架
- 对初学者更友好的入口是 **AgentChat**
- 更底层、更灵活的能力在 **autogen-core**
- 整体哲学偏“先把协作跑通，再逐步精细化”
- 很适合课堂演示、角色协作和快速原型

> 来自官方文档的稳定结论：AgentChat 建立在 Core 之上。

---

## AutoGen 的两层心智模型

| 层级 | 主要作用 | 更适合谁 |
| --- | --- | --- |
| AgentChat | 高层 API，快速做单/多 Agent 应用 | 初学者、原型阶段 |
| Core | 事件驱动运行时，控制更细 | 需要底层控制的工程团队 |
| Extensions | 模型、工具、外部服务连接层 | 需要接真实能力时 |

课堂建议：

- 第一轮先学 AgentChat
- 需要更强控制时再下钻 Core

---

## AutoGen 的核心对象与协作模型

- Agent：执行任务的主体
- Message：显式消息类型，承载上下文
- Tool：扩展 Agent 的外部能力
- Team：把多个 Agent 组织成协作单元
- Termination：决定何时收束对话或流程

最关键的是：

- 输入输出清晰
- 协作边界清晰
- 结束条件清晰

---

## AutoGen 的对话式协作

- 很多工作流以“对话”作为流程载体
- 角色扮演式协作非常适合教学
- 常见模式包括 Round-Robin、Selector、Swarm
- 对话让流程更直观，但也更容易跑散
- 不要把“多说几轮”误当成“更智能”

典型角色：

- Researcher
- Writer
- Critic / Editor

---

## AutoGen 的 Tool Use 与代码执行边界

- AgentChat 支持 tools、workbench 和外部能力扩展
- 工具能把“会说”变成“会做”
- 代码执行适合分析、脚本化和自动化验证
- 但读写权限、执行环境、超时必须被约束
- 最好把只读工具和高风险工具分开管理

> 讲师提示：这里一定要讲“沙箱”和白名单。

---

## AutoGen 的终止条件、回合控制与 Guardrails

- 多 Agent 最大风险之一是无穷对话
- 终止条件应覆盖步数、时间、成本和命中信号
- 可以用结构化输出、校验器和回调做 guardrails
- 对不确定结果，要有退出到人工或上层应用的路径
- 流程越开放，越需要清晰的 stopping rule

课堂要点：

- 先决定何时停止
- 再决定如何继续

---

## AutoGen 的 Trace、Debug 与 HITL

- 官方文档强调 logging、internal messages 与 state 管理
- Debug 时先看消息序列，再看工具调用，再看终止条件
- HITL 可以通过用户代理或外部中断点接入
- 保存和恢复 state 有助于多轮会话与复盘
- 调试顺序建议固定，避免盲猜模型问题

推荐排查顺序：

- prompt
- context
- tool result
- termination

---

## AutoGen 的优势与适用场景

- 上手快，适合教学和实验
- 很适合角色协作、研究助理、评审回路
- 适合快速验证“这套分工是否有价值”
- 当流程仍在探索期时，改动成本比较低
- 适合先用小范围 Demo 试流程

典型场景：

- 研究摘要
- 多角色写作
- 轻量 RAG

---

## AutoGen 的局限与不适合场景

- 如果流程需要极强的显式状态控制，它未必是第一选择
- 对话轮次一多，调试和成本都会上升
- 角色太多时，容易把简单问题复杂化
- 工具权限设计不好，风险会被放大
- 真正的长期业务流程，往往还要补外围编排层

一句话：

- 不要为了“多 Agent”而制造多轮对话

---

## AutoGen 最小原型、伪代码与教学点

```python
assistant = AssistantAgent("researcher", model_client=model)
editor = AssistantAgent("editor", model_client=model)
team = RoundRobinGroupChat(
    [assistant, editor],
    termination_condition=MaxMessageTermination(6),
)
result = await team.run(task="写一份带校验意见的主题摘要")
```

- 先用少角色 + 小上下文验证结构
- 课堂看点：角色边界、消息流、termination

> 📖 完整讲义已嵌入本节（原 `课程资料.md` / `Marp 幻灯片.md` 已删除）


---

## Part 2 — AutoGen 实战

> **AutoGen** (`autogen-agentchat`) 提供低层级的 Agent 控制，适合需要精确控制消息流与执行顺序的场景。
> OpenRouter 提供 OpenAI 兼容接口，通过 `OpenAIChatCompletionClient` 即可接入。

### 核心概念
- **`AssistantAgent`** — 由 LLM 驱动的 Agent，可绑定工具
- **`GraphFlow`** — 用有向图定义 Agent 间的消息传递顺序
- **`SelectorGroupChat`** — 由 LLM 动态选择下一个发言的 Agent
- **终止条件** — 可组合（`|` 运算符）：`TextMentionTermination`、`MaxMessageTermination`

In [ ]:
# AutoGen 公共导入
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow, SelectorGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

# 通过 OpenRouter 初始化 Claude 客户端（OpenAI 兼容接口）
sonnet_client = OpenAIChatCompletionClient(
    model=MODEL_SONNET,
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
    model_capabilities={
        'json_output': False,
        'vision': True,
        'function_calling': True,
        'multiple_system_messages': True,
    },
)

haiku_client = OpenAIChatCompletionClient(
    model=MODEL_HAIKU,
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
    model_capabilities={
        'json_output': False,
        'vision': True,
        'function_calling': True,
        'multiple_system_messages': True,
    },
)

print('✅ AutoGen OpenRouter 客户端初始化完成')
print(f'   Sonnet: {MODEL_SONNET}')
print(f'   Haiku : {MODEL_HAIKU}')

### 示例 2A — GraphFlow：研究员 → 写作者 → 审校者

**GraphFlow** 使用有向无环图（DAG）定义 Agent 执行顺序，适合流水线式协作。

```
Researcher ──► Writer ──► Editor
```

In [ ]:
# 定义三个角色 Agent
researcher = AssistantAgent(
    name='Researcher',
    model_client=haiku_client,
    system_message=(
        '你是一位专业研究员。当收到主题后，用 3-5 个要点总结该主题的关键信息，'
        '每个要点以「- 」开头，内容简洁准确。输出纯文本，不要使用标题。'
    ),
)

writer = AssistantAgent(
    name='Writer',
    model_client=sonnet_client,
    system_message=(
        '你是一位内容写作者。根据研究员提供的要点，撰写一篇结构清晰的 3 段式文章。'
        '段落分别为：引言、主体、结论。语言流畅，面向普通读者。'
    ),
)

editor = AssistantAgent(
    name='Editor',
    model_client=haiku_client,
    system_message=(
        '你是一位严格的审校者。审查写作者的文章，指出最多 3 处需要改进的地方，'
        '并给出修改建议。最后写「APPROVED」表示审校完成。'
    ),
)

# 构建有向图：Researcher → Writer → Editor
builder = DiGraphBuilder()
builder.add_node(researcher)
builder.add_node(writer)
builder.add_node(editor)
builder.add_edge(researcher, writer)
builder.add_edge(writer, editor)
graph = builder.build()

# 创建 GraphFlow 团队
termination = TextMentionTermination('APPROVED') | MaxMessageTermination(10)
pipeline_flow = GraphFlow(
    participants=[researcher, writer, editor],
    graph=graph,
    termination_condition=termination,
)

print('✅ GraphFlow 流水线构建完成：Researcher → Writer → Editor')

In [ ]:
# 运行流水线
TOPIC = '多 Agent 系统在企业自动化中的应用'

print(f'📋 任务主题：{TOPIC}\n')
print('=' * 60)

await Console(
    pipeline_flow.run_stream(
        task=f'请以「{TOPIC}」为主题，完成研究与写作任务。'
    )
)

### 示例 2B — SelectorGroupChat：动态 Agent 调度

**SelectorGroupChat** 在每一轮由 LLM 根据上下文选择最适合发言的 Agent。适合需要灵活调度的场景。

```
        ┌─────────────┐
        │  Selector   │  ◄── 由 LLM 决定下一个发言者
        └──────┬──────┘
    ┌──────────┼──────────┐
    ▼          ▼          ▼
 Analyst   Strategist  Critic
```

In [ ]:
# 定义三个具有不同专长的 Agent（带 description 供 Selector 参考）
analyst = AssistantAgent(
    name='Analyst',
    model_client=haiku_client,
    description='数据分析师，擅长分解问题、提供数据支持和量化分析',
    system_message=(
        '你是数据分析师。从数据和事实角度分析问题，提供量化依据和结构化分解。'
        '回复简洁，以要点形式呈现。'
    ),
)

strategist = AssistantAgent(
    name='Strategist',
    model_client=sonnet_client,
    description='战略规划师，擅长制定行动计划、评估风险和机会',
    system_message=(
        '你是战略规划师。基于分析结果提出具体可行的策略建议，'
        '评估风险与机会，给出优先级排序。'
    ),
)

critic = AssistantAgent(
    name='Critic',
    model_client=haiku_client,
    description='批判性思维专家，负责审查盲点、挑战假设、提出反驳',
    system_message=(
        '你是批判性思维专家。审查其他 Agent 的输出，识别潜在问题、'
        '逻辑漏洞和被忽视的假设。讨论完成后输出「TERMINATE」。'
    ),
)

# Selector 提示词：指导 LLM 如何选择下一个发言者
# AutoGen 0.7+ 支持的占位符：{roles}（名称+描述列表）、{participants}（名称列表）、{history}
selector_prompt = """
你是一个对话协调者。根据当前对话，选择最适合下一步发言的 Agent。

规则：
- 对话开始时，先让 Analyst 提供数据分析
- 有了分析基础后，让 Strategist 制定策略
- 有了策略后，让 Critic 进行批判性审查
- 如果 Critic 提出了新问题，可以重新调度 Analyst 或 Strategist 回应
- 避免同一个 Agent 连续发言两次

可用 Agent：
{roles}

对话历史：
{history}

下一个发言者（只输出 Agent 名称）：
"""

# 创建 SelectorGroupChat
selector_termination = TextMentionTermination('TERMINATE') | MaxMessageTermination(8)
selector_team = SelectorGroupChat(
    participants=[analyst, strategist, critic],
    model_client=sonnet_client,
    termination_condition=selector_termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=False,
)

print('✅ SelectorGroupChat 构建完成')

In [ ]:
print('📊 议题：企业是否应该在 2025 年大规模部署多 Agent AI 系统？\n')
print('=' * 60)

await Console(
    selector_team.run_stream(
        task='议题：企业是否应该在 2025 年大规模部署多 Agent AI 系统？请从各自角度深入讨论。'
    )
)

### 示例 2C — Memory：Agent 记忆

**`ListMemory`** 是最简单的内存类型，适合存储用户偏好、上下文信息等。

In [ ]:
from autogen_core.memory import ListMemory, MemoryContent, MemoryMimeType

# 创建用户记忆
user_memory = ListMemory()

# 向记忆中添加上下文信息
await user_memory.add(MemoryContent(
    content='用户是一名 AI 工程师，正在学习多 Agent 框架，偏好简洁的技术解释。',
    mime_type=MemoryMimeType.TEXT,
))
await user_memory.add(MemoryContent(
    content='用户已完成 LangChain 和 LangGraph 基础课程，熟悉异步编程。',
    mime_type=MemoryMimeType.TEXT,
))
await user_memory.add(MemoryContent(
    content='用户最感兴趣的是多 Agent 系统的生产化和可观测性。',
    mime_type=MemoryMimeType.TEXT,
))

# 创建带记忆的 Agent（直接运行，无需包装在 Team 中）
memory_agent = AssistantAgent(
    name='PersonalizedAssistant',
    model_client=sonnet_client,
    memory=[user_memory],
    system_message='你是一个个性化助手，能够根据用户背景提供定制化的技术建议。',
)

# 直接调用 Agent（单轮对话）
response = await memory_agent.run(
    task='我想学习如何监控多 Agent 系统，应该从哪里开始？'
)

print('💬 个性化回答：')
print(response.messages[-1].content)